# Week 1: Heart Failure Data Analysis

**Team:** Cameron and Julian  
**Goal:** Load UCI dataset and understand basic patterns  
**Dataset:** 299 heart failure patients, 12 clinical features

## 1. Load Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('default')
sns.set_palette('viridis')

print("Libraries loaded successfully")

## 2. Load Heart Failure Dataset

In [ ]:
# Load dataset
try:
    # Try multiple file locations
    for filepath in ['../original.csv', '../training_data.csv', 'original.csv']:
        try:
            df = pd.read_csv(filepath)
            print(f"Dataset loaded from: {filepath}")
            break
        except FileNotFoundError:
            continue
    
    # Display basic info
    print(f"Dataset shape: {df.shape}")
    print(f"Features: {list(df.columns)}")
    
except Exception as e:
    print(f"Error loading dataset: {e}")

## 3. Basic Dataset Information

In [ ]:
# Dataset overview
print("DATASET OVERVIEW")
print("="*40)
print(f"Total patients: {len(df)}")
print(f"Features: {len(df.columns)}")
print(f"Missing values: {df.isnull().sum().sum()}")

# Target variable analysis
if 'DEATH_EVENT' in df.columns:
    deaths = df['DEATH_EVENT'].sum()
    survivors = len(df) - deaths
    survival_rate = survivors / len(df)
    
    print(f"\nOUTCOME DISTRIBUTION:")
    print(f"Survivors: {survivors} ({survival_rate:.1%})")
    print(f"Deaths: {deaths} ({1-survival_rate:.1%})")

# Display first few rows
print("\nFIRST 5 ROWS:")
df.head()

## 4. Statistical Summary

In [ ]:
# Basic statistics
print("STATISTICAL SUMMARY")
print("="*40)
df.describe()

## 5. Correlation Analysis

In [ ]:
# Calculate correlations with death events
if 'DEATH_EVENT' in df.columns:
    correlations = df.corr()['DEATH_EVENT'].sort_values(key=abs, ascending=False)
    
    print("CORRELATIONS WITH DEATH EVENTS")
    print("="*40)
    for feature, corr in correlations.items():
        if feature != 'DEATH_EVENT':
            print(f"{feature}: {corr:.3f}")
    
    # Create correlation heatmap
    plt.figure(figsize=(12, 10))
    sns.heatmap(df.corr(), annot=True, cmap='coolwarm', center=0, 
                fmt='.2f', square=True)
    plt.title('Feature Correlation Heatmap')
    plt.tight_layout()
    plt.show()

## 6. Key Visualizations

In [ ]:
# Create comprehensive visualization dashboard
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. Survival distribution
if 'DEATH_EVENT' in df.columns:
    survival_counts = df['DEATH_EVENT'].value_counts()
    axes[0,0].pie(survival_counts.values, labels=['Survived', 'Died'], 
                  autopct='%1.1f%%', colors=['lightgreen', 'lightcoral'])
    axes[0,0].set_title('Survival Distribution')

# 2. Age distribution by outcome
if 'age' in df.columns and 'DEATH_EVENT' in df.columns:
    for outcome in [0, 1]:
        subset = df[df['DEATH_EVENT'] == outcome]['age']
        label = 'Survived' if outcome == 0 else 'Died'
        axes[0,1].hist(subset, alpha=0.7, label=label, bins=15)
    axes[0,1].set_xlabel('Age')
    axes[0,1].set_ylabel('Count')
    axes[0,1].set_title('Age Distribution by Outcome')
    axes[0,1].legend()

# 3. Ejection fraction by outcome
if 'ejection_fraction' in df.columns and 'DEATH_EVENT' in df.columns:
    df.boxplot(column='ejection_fraction', by='DEATH_EVENT', ax=axes[0,2])
    axes[0,2].set_title('Ejection Fraction by Outcome')
    axes[0,2].set_xlabel('Death Event')

# 4. Serum creatinine by outcome
if 'serum_creatinine' in df.columns and 'DEATH_EVENT' in df.columns:
    df.boxplot(column='serum_creatinine', by='DEATH_EVENT', ax=axes[1,0])
    axes[1,0].set_title('Serum Creatinine by Outcome')
    axes[1,0].set_xlabel('Death Event')

# 5. Time distribution
if 'time' in df.columns:
    axes[1,1].hist(df['time'], bins=20, alpha=0.7, color='skyblue')
    axes[1,1].set_xlabel('Follow-up Time (days)')
    axes[1,1].set_ylabel('Count')
    axes[1,1].set_title('Follow-up Time Distribution')

# 6. Top correlations
if 'DEATH_EVENT' in df.columns:
    top_features = ['ejection_fraction', 'serum_creatinine', 'age', 'time']
    corr_values = []
    feature_names = []
    
    for feature in top_features:
        if feature in df.columns:
            corr = abs(df[feature].corr(df['DEATH_EVENT']))
            corr_values.append(corr)
            feature_names.append(feature)
    
    axes[1,2].barh(feature_names, corr_values)
    axes[1,2].set_xlabel('Absolute Correlation with Death')
    axes[1,2].set_title('Feature Importance Preview')

plt.suptitle('Heart Failure Dataset - Week 1 Analysis', fontsize=16)
plt.tight_layout()
plt.show()

## 7. Key Findings Summary

In [ ]:
# Summarize key findings
print("WEEK 1 KEY FINDINGS")
print("="*40)

print(f"1. Dataset loaded successfully: {len(df)} patients, {len(df.columns)} features")
print(f"2. No missing values detected")

if 'DEATH_EVENT' in df.columns:
    survival_rate = 1 - df['DEATH_EVENT'].mean()
    print(f"3. Overall survival rate: {survival_rate:.1%}")
    
    # Top predictive features
    correlations = df.corr()['DEATH_EVENT'].abs().sort_values(ascending=False)
    print(f"4. Most predictive features:")
    for i, (feature, corr) in enumerate(correlations.head(4).items(), 1):
        if feature != 'DEATH_EVENT':
            print(f"   {i-1}. {feature} (correlation: {df[feature].corr(df['DEATH_EVENT']):.3f})")

print("\nREADY FOR WEEK 2: Machine Learning Model Development")
print("Next: Juan and Nathan will build predictive models")

## 8. Data Export for Modeling

In [ ]:
# Save cleaned data for next week
try:
    df.to_csv('clean_heart_failure_data.csv', index=False)
    print("Clean dataset saved as 'clean_heart_failure_data.csv'")
    print("Ready for Week 2 modeling phase")
except Exception as e:
    print(f"Error saving data: {e}")

# Summary statistics for handoff
summary_stats = {
    'total_patients': len(df),
    'total_features': len(df.columns),
    'missing_values': df.isnull().sum().sum(),
    'survival_rate': 1 - df['DEATH_EVENT'].mean() if 'DEATH_EVENT' in df.columns else None
}

print("\nHANDOFF SUMMARY FOR WEEK 2:")
for key, value in summary_stats.items():
    print(f"{key}: {value}")